# Create the recommendation training dataset

This notebook prepares labeled user and business examples for the business recommendation model. Run the cells from top to bottom.

## 1. Set up paths and imports

The project folder is located automatically, whether Jupyter starts in the project root or in this notebook folder.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
BASE_DIR = next((parent for parent in (cwd, *cwd.parents) if (parent / 'data' / 'business_recommendation_dataset.csv').exists()), None)
if BASE_DIR is None:
    BASE_DIR = next((parent / 'recommendationmodel' for parent in (cwd, *cwd.parents) if (parent / 'recommendationmodel' / 'data' / 'business_recommendation_dataset.csv').exists()), None)
if BASE_DIR is None:
    raise FileNotFoundError('Could not locate recommendationmodel/data/business_recommendation_dataset.csv')

DATA_DIR = BASE_DIR / 'data'
BUSINESS_PATH = DATA_DIR / 'business_recommendation_dataset.csv'
TRAINING_PATH = DATA_DIR / 'training_data.csv'
print(f'Project folder: {BASE_DIR}')

## 2. Load and inspect the business catalog

Each business supplies investment, time, skill, land, experience, and demand requirements used to create training examples.

In [ ]:
businesses = pd.read_csv(BUSINESS_PATH)
print(f'Loaded {len(businesses)} businesses.')
businesses.head()

In [ ]:
print('Dataset shape:', businesses.shape)
print('Columns:', businesses.columns.tolist())
businesses.isna().sum()

## 3. Check the columns needed for scoring

Stop early with a readable message if the catalog is missing a field required below.

In [ ]:
required = {'business', 'min_investment', 'max_investment', 'required_time', 'skill', 'land_required', 'experience_required', 'demand'}
missing = required - set(businesses.columns)
if missing:
    raise ValueError(f'Missing business columns: {sorted(missing)}')

businesses = businesses.dropna(subset=['business', 'skill']).copy()
print(f'{len(businesses)} businesses are ready for profile generation.')

## 4. Generate example user profiles

For each business, create a variety of possible users. Most profiles have a different skill from the business so the training data includes both good and poor matches. A fixed random seed makes the generated data reproducible.

In [ ]:
rng = np.random.default_rng(42)
examples_per_business = 250
rows = []
skills = businesses['skill'].dropna().astype(str).unique()

for business in businesses.to_dict(orient='records'):
    for _ in range(examples_per_business):
        investment = int(rng.integers(20_000, 500_001))
        available_time = int(rng.integers(1, 13))
        if rng.random() < 0.7:
            skill = str(business['skill'])
        else:
            skill = str(rng.choice(skills))
        land = int(rng.integers(0, 2))
        experience = int(rng.integers(0, 11))

        rows.append({
            'investment': investment,
            'available_time': available_time,
            'skill': skill,
            'land': land,
            'experience': experience,
            'business': business['business'],
            'min_investment': business['min_investment'],
            'max_investment': business['max_investment'],
            'required_time': business['required_time'],
            'required_skill': business['skill'],
            'land_required': business['land_required'],
            'experience_required': business['experience_required'],
            'demand': business['demand'],
        })

profiles = pd.DataFrame(rows)
print(f'Created {len(profiles):,} user-business examples.')
profiles.head()

## 5. Score how well each profile fits

The score combines budget, time, skill, land, experience, and business demand. Stronger matches receive higher scores between 0 and 1.

In [ ]:
investment_fit = profiles['investment'].between(profiles['min_investment'], profiles['max_investment'])
time_fit = profiles['available_time'] >= profiles['required_time']
skill_fit = profiles['skill'].str.strip().str.lower() == profiles['required_skill'].str.strip().str.lower()
land_fit = (profiles['land_required'] == 0) | (profiles['land'] == 1)
experience_fit = profiles['experience'] >= profiles['experience_required']
demand_score = (profiles['demand'] / 5).clip(0, 1)

profiles['score'] = (
    0.35 * investment_fit.astype(float)
    + 0.20 * time_fit.astype(float)
    + 0.20 * skill_fit.astype(float)
    + 0.10 * land_fit.astype(float)
    + 0.10 * experience_fit.astype(float)
    + 0.05 * demand_score
).round(4)

profiles[['business', 'investment', 'available_time', 'skill', 'score']].head(10)

## 6. Keep the model training columns

The recommender expects these six input features and the numeric `score` target. The business requirement columns above were only used to calculate the label.

In [ ]:
training_data = profiles[[
    'investment', 'available_time', 'skill', 'land',
    'experience', 'business', 'score'
]].copy()
training_data.head()

## 7. Review and save the dataset

Save the CSV in the project data folder. The next notebook, `train_model.ipynb`, uses this file.

In [ ]:
print('Training dataset shape:', training_data.shape)
print('Missing values:')
display(training_data.isna().sum())
print('Score summary:')
display(training_data['score'].describe())

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
training_data.to_csv(TRAINING_PATH, index=False)
print(f'Saved {len(training_data):,} rows to {TRAINING_PATH}')

In [ ]:
saved_data = pd.read_csv(TRAINING_PATH)
print(f'Reloaded dataset: {saved_data.shape[0]:,} rows, {saved_data.shape[1]} columns')
saved_data.head()